# Transformer geDIG Reproduction Experiment

**Purpose**: Investigate the Phase 3 vs Phase 5 contradiction

| Phase | Method | F Change | Accuracy Change |
|-------|--------|----------|----------------|
| Phase 3 | Forced attention edit | F down | 97% -> 47% |
| Phase 5 | F regularization | F down | 86.0% -> 86.3% |

**Hypothesis**: The difference is due to intervention strength, not F itself.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository
import os
import subprocess

if not os.path.isdir('InsightSpike-AI'):
    subprocess.check_call(['git', 'clone', 'https://github.com/miyauchikazuyoshi/InsightSpike-AI.git'])
%cd InsightSpike-AI
!git pull origin main

In [ ]:
# Install dependencies
!pip -q install -U pip
!pip -q install transformers datasets accelerate networkx scipy torch

## Phase 1: Basic Validation

Verify that extract_and_score.py works and F_real < F_random.

In [ ]:
# Run basic extraction (BERT + GPT-2, 16 texts, 4 layers)
!python experiments/transformer/extract_and_score.py \
    --text-count 16 \
    --layer-cap 4 \
    --device auto \
    --out experiments/transformer/results/phase1/score_basic.json

In [ ]:
# Analyze results
import json
import numpy as np
from pathlib import Path

data = json.loads(Path('experiments/transformer/results/phase1/score_basic.json').read_text())
rows = [r for r in data if not r.get('subgraph')]

f_real = np.array([r['F'] for r in rows])
f_random = np.array([r['baseline_F_random'] for r in rows])
delta = f_real - f_random

print(f"Rows: {len(rows)}")
print(f"F_real mean: {f_real.mean():.4f}")
print(f"F_random mean: {f_random.mean():.4f}")
print(f"Delta mean: {delta.mean():.4f} (positive = real > random)")
print(f"Win rate (F_real > F_random): {(delta > 0).mean():.1%}")

## Phase 2: Threshold Sensitivity Analysis

In [ ]:
# Sweep different percentile thresholds
import subprocess

for pct in [0.80, 0.85, 0.90, 0.95]:
    cmd = f"""python experiments/transformer/extract_and_score.py \
        --text-count 16 \
        --layer-cap 4 \
        --percentile {pct} \
        --device auto \
        --out experiments/transformer/results/phase2/score_pct{int(pct*100)}.json"""
    print(f"Running percentile={pct}...")
    !{cmd}

In [ ]:
# Compare threshold results
import matplotlib.pyplot as plt

results = {}
for pct in [80, 85, 90, 95]:
    path = Path(f'experiments/transformer/results/phase2/score_pct{pct}.json')
    if path.exists():
        data = json.loads(path.read_text())
        rows = [r for r in data if not r.get('subgraph')]
        f_real = np.array([r['F'] for r in rows])
        f_random = np.array([r['baseline_F_random'] for r in rows])
        results[pct] = {
            'f_real_mean': f_real.mean(),
            'f_random_mean': f_random.mean(),
            'delta_mean': (f_real - f_random).mean(),
            'win_rate': (f_real > f_random).mean()
        }

# Plot
pcts = list(results.keys())
deltas = [results[p]['delta_mean'] for p in pcts]
win_rates = [results[p]['win_rate'] for p in pcts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(pcts, deltas)
ax1.set_xlabel('Percentile Threshold')
ax1.set_ylabel('Delta (F_real - F_random)')
ax1.set_title('Threshold vs Delta')
ax1.axhline(0, color='red', linestyle='--')

ax2.bar(pcts, win_rates)
ax2.set_xlabel('Percentile Threshold')
ax2.set_ylabel('Win Rate')
ax2.set_title('Threshold vs Win Rate')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('experiments/transformer/results/phase2/threshold_sweep.png', dpi=150)
plt.show()

## Phase 3: Intervention Strength Sweep

Test different intervention strengths to find the threshold where accuracy drops.

In [ ]:
# Run intervention experiments with varying strengths
!python experiments/transformer/intervene_eval.py

In [ ]:
# Analyze intervention results
path = Path('results/transformer_gedig/intervene_sketch.json')
if path.exists():
    data = json.loads(path.read_text())
    
    print("=== SST2 Baseline ===")
    print(f"Accuracy: {data['sst2_eval']['accuracy']:.1%}")
    print(f"F vs Confidence correlation: {data['sst2_eval']['conf_vs_F_corr']:.3f}")
    
    print("\n=== Interventions ===")
    for name, result in data['sst2_interventions'].items():
        print(f"{name}: Accuracy={result['accuracy']:.1%}")

## Phase 5: F-Regularization Reproduction

Reproduce the F-regularization experiment to verify the results.

In [ ]:
# Quick smoke test
!python experiments/transformer/train_f_regularized.py \
    --alpha 0.001 \
    --train-samples 500 \
    --eval-samples 200 \
    --epochs 2 \
    --output-dir experiments/transformer/results/f_reg_smoke

In [ ]:
# Full alpha sweep (takes longer)
!python experiments/transformer/train_f_regularized.py \
    --alpha-sweep \
    --alphas "0,0.001,0.01,0.1" \
    --seeds "42,123" \
    --train-samples 2000 \
    --eval-samples 500 \
    --epochs 3 \
    --output-dir experiments/transformer/results/f_reg_sweep

## Save Results to Drive

In [ ]:
import shutil

save_dir = Path('/content/drive/MyDrive/insightspike/transformer_experiments')
save_dir.mkdir(parents=True, exist_ok=True)

# Copy results
results_dir = Path('experiments/transformer/results')
if results_dir.exists():
    for f in results_dir.rglob('*'):
        if f.is_file():
            dest = save_dir / f.relative_to(results_dir)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            print(f'Saved: {dest}')

print('\nDone! Results saved to:', save_dir)